In [ ]:
%%sql -r NAGOROFILES_TXT_PARSE
CREATE OR REPLACE TABLE TME_DB.ACT1.NAGORO_DOCUMENTS AS
SELECT
    RELATIVE_PATH AS file_name,
    AI_PARSE_DOCUMENT(
        TO_FILE('@TME_DB.ACT1.NAGOROFILES', RELATIVE_PATH),
        {'mode': 'LAYOUT'}
    ):content::VARCHAR AS content
FROM DIRECTORY(@TME_DB.ACT1.NAGOROFILES)
WHERE RELATIVE_PATH LIKE '%.txt'

In [ ]:
%%sql -r NAGORO_SEARCH_PARSER
CREATE OR REPLACE CORTEX SEARCH SERVICE TME_DB.ACT1.NAGORO_SEARCH_SERVICE
  ON content
  ATTRIBUTES file_name
  WAREHOUSE = TME_WH_XS
  TARGET_LAG = '1 hour'
AS (
  SELECT content, file_name
  FROM TME_DB.ACT1.NAGORO_DOCUMENTS
)

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE STAGE TME_DB.ACT1.ACT1_LORE_SSE ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') DIRECTORY = (ENABLE = TRUE) COMMENT = 'This stage documents the storyline behind what happened in ACT 1, with txt files.'

In [ ]:
%%sql -r dataframe_5
COPY FILES INTO @TME_DB.ACT1.ACT1_LORE_SSE FROM @TME_DB.ACT1.ACT1_LORE

In [ ]:
%%sql -r dataframe_6
ALTER STAGE TME_DB.ACT1.ACT1_LORE_SSE REFRESH

In [ ]:
%%sql -r ACT1_LOREREADER
CREATE OR REPLACE TABLE TME_DB.ACT1.ACT1_LOREREADER AS
SELECT
    RELATIVE_PATH AS file_name,
    AI_PARSE_DOCUMENT(
        TO_FILE('@TME_DB.ACT1.ACT1_LORE_SSE', RELATIVE_PATH),
        {'mode': 'LAYOUT'}
    ):content::VARCHAR AS content
FROM DIRECTORY(@TME_DB.ACT1.ACT1_LORE_SSE)
WHERE RELATIVE_PATH LIKE '%.txt'

In [ ]:
%%sql -r ACT1_LOREPROCESSER
CREATE OR REPLACE CORTEX SEARCH SERVICE TME_DB.ACT1.ACT1_SEARCH_SERVICE
  ON content
  ATTRIBUTES file_name
  WAREHOUSE = TME_WH_XS
  TARGET_LAG = '1 hour'
AS (
  SELECT content, file_name
  FROM TME_DB.ACT1.ACT1_LOREREADER
)

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE TABLE TME_DB.ACT1.ACT1_LOREREADER_CHUNKED AS
SELECT
    file_name,
    content AS full_content,
    c.value::VARCHAR AS chunk
FROM TME_DB.ACT1.ACT1_LOREREADER,
    LATERAL FLATTEN(input => SPLIT(content, '\n\n')) c
WHERE c.value::VARCHAR <> '';

CREATE OR REPLACE CORTEX SEARCH SERVICE TME_DB.ACT1.ACT1_SEARCH_SERVICE
  ON chunk
  ATTRIBUTES file_name
  WAREHOUSE = TME_WH_XS
  TARGET_LAG = '1 hour'
AS (
  SELECT chunk, file_name, full_content
  FROM TME_DB.ACT1.ACT1_LOREREADER_CHUNKED
);